In [3]:
from dataclasses import dataclass
from typing import Dict
import math

In [4]:
@dataclass
class Decision:
  allowed:bool
  remaining:int
  retry_after_ms:int

In [5]:
@dataclass
class BucketState:
  tokens:float
  last_refill_ms:int

In [9]:
from re import S
class TokenBucketRateLimiter:
  def __init__(self,capacity:int,refill_rate:float):
    self.capacity=capacity
    self.refill_rate=refill_rate
    self.buckets:Dict[str,BucketState]={}

  def _refill(self,state:BucketState, current_time_ms:int) -> None:
    elapsed_ms=current_time_ms-state.last_refill_ms
    if elapsed_ms>0:
      state.tokens=min(self.capacity,state.tokens+(elapsed_ms/1000.0) * self.refill_rate, )
      state.last_refill_ms=current_time_ms

  def check(self,customer_id:str,current_time_ms:int) -> Decision:
    if customer_id not in self.buckets:
      self.buckets[customer_id]=BucketState(tokens=float(self.capacity),last_refill_ms=current_time_ms)
    state=self.buckets[customer_id]
    self._refill(state,current_time_ms)
    if state.tokens>=1.0:
      state.tokens-=1.0
      return Decision(True,int(state.tokens),0)
    tokens_needed=1.0-state.tokens
    seconds_needed=tokens_needed/self.refill_rate
    retry_after_ms=int(math.ceil(seconds_needed*1000))
    return Decision(False,int(state.tokens),retry_after_ms)


In [12]:
limiter=TokenBucketRateLimiter(100,10)
result=limiter.check("stripe-test",0)
print(result)

Decision(allowed=True, remaining=99, retry_after_ms=0)
